# Cheminformatics Workflow

This notebook shows the recommended chemistry workflow in TMAP.

We will use `examples/cluster_65053.csv` and go from SMILES to an interactive molecular map.


## 1. Load Molecules


In [1]:
import pandas as pd

df = pd.read_csv("../examples/cluster_65053.csv")
smiles = df["smiles"].tolist()

print(f"Loaded {len(smiles):,} molecules")


Loaded 64,098 molecules


## 2. Compute Fingerprints, Properties, And Scaffolds


In [2]:
from tmap.utils import fingerprints_from_smiles, molecular_properties, murcko_scaffolds

fps, valid = fingerprints_from_smiles(
    smiles, fp_type="morgan", radius=2, n_bits=2048, return_valid=True
)
smiles = [s for s, ok in zip(smiles, valid) if ok]  # keep labels aligned with fingerprint rows

props = molecular_properties(smiles, properties=["mw", "logp", "n_rings", "qed"])
scaffolds = murcko_scaffolds(smiles)

print(fps.shape)
print(sorted(props))


  [Props] 64,098 done, 0 invalid
  [Scaffolds] 64,098/64,098 valid
(64098, 2048)
['logp', 'mw', 'n_rings', 'qed']


## 3. Fit TMAP


In [3]:
from tmap import TMAP

# Dense 0/1 fingerprints go to the USearch Jaccard index.
# n_permutations and kc only apply to sets, strings and sparse input (MinHash + LSHForest).
model = TMAP(metric="jaccard", n_neighbors=20, seed=42).fit(fps)

print(model.embedding_.shape)
print(model.tree_.edges.shape)


(64098, 2)
(64097, 2)


## 4. Build A Molecular Visualization


In [4]:
viz = model.to_tmapviz()
viz.title = "Cluster 65053"
viz.add_smiles(smiles)
viz.add_color_layout("MW", props["mw"].tolist(), color="viridis")
viz.add_color_layout("LogP", props["logp"].tolist(), color="plasma")
viz.add_color_layout("Ring Count", props["n_rings"].tolist(), categorical=True, color="tab10")
viz.add_color_layout("QED", props["qed"].tolist(), color="magma")
viz.add_label("Murcko Scaffold", scaffolds.tolist())


## 5. Explore In Jupyter

The widget draws points only and warns that the tree edges are ignored. The HTML file in the next step includes them.


In [5]:
widget = viz.to_widget(width=1000, height=650, controls=True)
widget.show()


## 6. Save HTML


In [6]:
path = viz.write_html("cluster_65053.html")
print(path)


cluster_65053.html


## 7. Serve A Larger Version Locally


In [7]:
# This starts a local server and blocks the notebook cell until you stop it.
viz.serve(port=8050)


Serving TMAP visualization at http://127.0.0.1:8050
  64,098 points, 64,097 edges
  Press Ctrl+C to stop.

Shutting down server...
